<a href="https://colab.research.google.com/github/MahrushSalam/pbo/blob/main/mproj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from datetime import datetime

# 1. DECORATOR

def log_aktivitas(func):
    """
    Decorator untuk mencatat aktivitas method.
    """
    def wrapper(*args, **kwargs):
        print(f"\n[LOG] Menjalankan: {func.__name__}")
        hasil = func(*args, **kwargs)
        print(f"[LOG] Selesai: {func.__name__}")
        return hasil

    return wrapper

# 2. CLASS KENDARAAN

class Kendaraan:
    """
    Parent class untuk seluruh jenis kendaraan.
    """

    # CLASS ATTRIBUTE
    jumlah_kendaraan = 0

    def __init__(self, id_kendaraan, merk, model, tahun, harga_sewa):
        # INSTANCE ATTRIBUTE
        self.id_kendaraan = id_kendaraan
        self.merk = merk
        self.model = model
        self.tahun = tahun
        self.harga_sewa = harga_sewa

        # PROTECTED ATTRIBUTE
        self._tersedia = True

        # PRIVATE ATTRIBUTE
        self.__total_disewa = 0

        Kendaraan.jumlah_kendaraan += 1

    # PUBLIC METHOD
    def tampilkan_info(self):
        status = "Tersedia" if self._tersedia else "Disewa"

        print(f"""
ID Kendaraan : {self.id_kendaraan}
Merk         : {self.merk}
Model        : {self.model}
Tahun        : {self.tahun}
Harga/Hari   : Rp{self.harga_sewa:,.0f}
Status       : {status}
""")

    # PUBLIC METHOD
    def sewa(self):
        if not self._tersedia:
            return False

        self._tersedia = False
        self.__total_disewa += 1
        return True

    # PUBLIC METHOD
    def kembalikan(self):
        self._tersedia = True

    # PUBLIC METHOD
    def cek_ketersediaan(self):
        return self._tersedia

    # PRIVATE DATA DIAMBIL MELALUI METHOD PUBLIC
    def jumlah_disewa(self):
        return self.__total_disewa

    # POLYMORPHISM
    def hitung_harga(self, lama_sewa):
        return self.harga_sewa * lama_sewa

    # DECORATOR
    @property
    def status(self):
        return "Tersedia" if self._tersedia else "Disewa"

# 3. INHERITANCE - CLASS MOBIL

class Mobil(Kendaraan):

    def __init__(
        self,
        id_kendaraan,
        merk,
        model,
        tahun,
        harga_sewa,
        jumlah_kursi
    ):
        super().__init__(
            id_kendaraan,
            merk,
            model,
            tahun,
            harga_sewa
        )

        self.jumlah_kursi = jumlah_kursi

    # POLYMORPHISM
    def hitung_harga(self, lama_sewa):
        harga_dasar = self.harga_sewa * lama_sewa

        # Diskon mobil jika disewa >= 7 hari
        if lama_sewa >= 7:
            diskon = harga_dasar * 0.10
            return harga_dasar - diskon

        return harga_dasar

    def tampilkan_info(self):
        super().tampilkan_info()
        print(f"Jumlah Kursi : {self.jumlah_kursi}")

# 4. INHERITANCE - CLASS MOTOR

class Motor(Kendaraan):

    def __init__(
        self,
        id_kendaraan,
        merk,
        model,
        tahun,
        harga_sewa,
        jenis_motor
    ):
        super().__init__(
            id_kendaraan,
            merk,
            model,
            tahun,
            harga_sewa
        )

        self.jenis_motor = jenis_motor

    # POLYMORPHISM
    def hitung_harga(self, lama_sewa):
        harga_dasar = self.harga_sewa * lama_sewa

        # Diskon motor jika disewa >= 5 hari
        if lama_sewa >= 5:
            diskon = harga_dasar * 0.05
            return harga_dasar - diskon

        return harga_dasar

    def tampilkan_info(self):
        super().tampilkan_info()
        print(f"Jenis Motor  : {self.jenis_motor}")

# 5. CLASS CUSTOMER

class Customer:

    # CLASS ATTRIBUTE
    jumlah_customer = 0

    def __init__(self, id_customer, nama, no_hp, alamat):
        # INSTANCE ATTRIBUTE
        self.id_customer = id_customer
        self.nama = nama
        self.no_hp = no_hp
        self.alamat = alamat

        Customer.jumlah_customer += 1

    # PUBLIC METHOD
    def tampilkan_info(self):
        print(f"""
ID Customer : {self.id_customer}
Nama        : {self.nama}
No. HP      : {self.no_hp}
Alamat      : {self.alamat}
""")

# 6. CLASS PEMBAYARAN

class Pembayaran:

    def __init__(self, metode, jumlah):
        self.metode = metode
        self.jumlah = jumlah
        self.status = "Belum Dibayar"

    @log_aktivitas
    def proses_pembayaran(self):
        self.status = "Lunas"

        print(f"""
Metode Pembayaran : {self.metode}
Jumlah Pembayaran : Rp{self.jumlah:,.0f}
Status            : {self.status}
""")

        return True

# 7. CLASS DETAIL RENTAL

class DetailRental:
    """
    CLASS INI DIGUNAKAN DALAM KOMPOSISI.

    DetailRental dibuat sebagai bagian dari Rental.
    """

    def __init__(self, kendaraan, lama_sewa):
        self.kendaraan = kendaraan
        self.lama_sewa = lama_sewa
        self.harga = kendaraan.hitung_harga(lama_sewa)

    def tampilkan_detail(self):
        print(f"""
Kendaraan : {self.kendaraan.merk} {self.kendaraan.model}
Durasi    : {self.lama_sewa} hari
Harga     : Rp{self.harga:,.0f}
""")

# 8. CLASS RENTAL

class Rental:

    # CLASS ATTRIBUTE
    biaya_admin = 10000

    def __init__(self, id_rental, customer):
        self.id_rental = id_rental
        self.customer = customer

        # KOMPOSISI
        self.detail_rental = None

        self.pembayaran = None
        self.tanggal = datetime.now()

    # AGREGASI
    # Rental menggunakan object Customer,
    # tetapi Customer tetap dapat berdiri sendiri.
    @log_aktivitas
    def tambah_kendaraan(self, kendaraan, lama_sewa):

        if not kendaraan.cek_ketersediaan():
            print("Kendaraan sedang tidak tersedia.")
            return False

        if lama_sewa <= 0:
            print("Lama sewa harus lebih dari 0 hari.")
            return False

        kendaraan.sewa()

        # KOMPOSISI
        self.detail_rental = DetailRental(
            kendaraan,
            lama_sewa
        )

        return True

    def hitung_total(self):

        if self.detail_rental is None:
            return 0

        return (
            self.detail_rental.harga
            + Rental.biaya_admin
        )

    @log_aktivitas
    def proses_pembayaran(self, metode):

        total = self.hitung_total()

        self.pembayaran = Pembayaran(
            metode,
            total
        )

        return self.pembayaran.proses_pembayaran()

    def tampilkan_struk(self):

        print("\n")
        print("=" * 55)
        print("              STRUK RENTAL KENDARAAN")
        print("=" * 55)

        print(f"ID Rental   : {self.id_rental}")
        print(
            f"Tanggal     : "
            f"{self.tanggal.strftime('%d-%m-%Y %H:%M')}"
        )

        print("\n--- DATA CUSTOMER ---")

        print(f"ID Customer : {self.customer.id_customer}")
        print(f"Nama        : {self.customer.nama}")
        print(f"No. HP      : {self.customer.no_hp}")

        if self.detail_rental:

            print("\n--- DETAIL KENDARAAN ---")

            print(
                f"Kendaraan   : "
                f"{self.detail_rental.kendaraan.merk} "
                f"{self.detail_rental.kendaraan.model}"
            )

            print(
                f"Lama Sewa   : "
                f"{self.detail_rental.lama_sewa} hari"
            )

            print(
                f"Harga Sewa  : "
                f"Rp{self.detail_rental.harga:,.0f}"
            )

            print(
                f"Biaya Admin : "
                f"Rp{Rental.biaya_admin:,.0f}"
            )

            print(
                f"TOTAL       : "
                f"Rp{self.hitung_total():,.0f}"
            )

        if self.pembayaran:

            print("\n--- PEMBAYARAN ---")

            print(
                f"Metode      : "
                f"{self.pembayaran.metode}"
            )

            print(
                f"Status      : "
                f"{self.pembayaran.status}"
            )

        print("=" * 55)

# 9. CLASS SISTEM RENTAL

class SistemRental:

    def __init__(self):

        # INSTANCE ATTRIBUTE

        # COLLECTION OBJECT
        self.daftar_kendaraan = []
        self.daftar_customer = []
        self.daftar_rental = []

        self.nomor_rental = 1

    # CUSTOMER

    def tambah_customer(self):

        print("\n===== TAMBAH CUSTOMER =====")

        id_customer = f"C{len(self.daftar_customer) + 1:03d}"

        nama = input("Nama       : ")
        no_hp = input("No. HP     : ")
        alamat = input("Alamat     : ")

        customer = Customer(
            id_customer,
            nama,
            no_hp,
            alamat
        )

        self.daftar_customer.append(customer)

        print(
            f"\nCustomer berhasil ditambahkan "
            f"dengan ID {id_customer}."
        )

    def tampilkan_customer(self):

        print("\n===== DAFTAR CUSTOMER =====")

        if not self.daftar_customer:
            print("Belum ada customer.")
            return

        for customer in self.daftar_customer:

            print("-" * 40)

            customer.tampilkan_info()

    # KENDARAAN

    def tambah_mobil(self):

        print("\n===== TAMBAH MOBIL =====")

        id_kendaraan = f"K{len(self.daftar_kendaraan) + 1:03d}"

        merk = input("Merk          : ")
        model = input("Model         : ")
        tahun = int(input("Tahun         : "))
        harga = float(input("Harga/hari    : "))
        kursi = int(input("Jumlah kursi  : "))

        mobil = Mobil(
            id_kendaraan,
            merk,
            model,
            tahun,
            harga,
            kursi
        )

        self.daftar_kendaraan.append(mobil)

        print(
            f"\nMobil berhasil ditambahkan "
            f"dengan ID {id_kendaraan}."
        )

    def tambah_motor(self):

        print("\n===== TAMBAH MOTOR =====")

        id_kendaraan = f"K{len(self.daftar_kendaraan) + 1:03d}"

        merk = input("Merk          : ")
        model = input("Model         : ")
        tahun = int(input("Tahun         : "))
        harga = float(input("Harga/hari    : "))
        jenis = input("Jenis motor   : ")

        motor = Motor(
            id_kendaraan,
            merk,
            model,
            tahun,
            harga,
            jenis
        )

        self.daftar_kendaraan.append(motor)

        print(
            f"\nMotor berhasil ditambahkan "
            f"dengan ID {id_kendaraan}."
        )

    def tampilkan_kendaraan(self):

        print("\n===== DAFTAR KENDARAAN =====")

        if not self.daftar_kendaraan:
            print("Belum ada kendaraan.")
            return

        for kendaraan in self.daftar_kendaraan:

            print("-" * 50)

            kendaraan.tampilkan_info()

    # CARI OBJECT

    def cari_customer(self, id_customer):

        for customer in self.daftar_customer:

            if customer.id_customer.lower() == id_customer.lower():
                return customer

        return None

    def cari_kendaraan(self, id_kendaraan):

        for kendaraan in self.daftar_kendaraan:

            if kendaraan.id_kendaraan.lower() == id_kendaraan.lower():
                return kendaraan

        return None

    # RENTAL

    @log_aktivitas
    def buat_rental(self):

        print("\n===== PEMBUATAN RENTAL =====")

        if not self.daftar_customer:
            print("Belum ada customer.")
            return

        if not self.daftar_kendaraan:
            print("Belum ada kendaraan.")
            return

        self.tampilkan_customer()

        id_customer = input(
            "\nMasukkan ID Customer: "
        )

        customer = self.cari_customer(id_customer)

        if customer is None:

            print("Customer tidak ditemukan.")

            return

        self.tampilkan_kendaraan()

        id_kendaraan = input(
            "\nMasukkan ID Kendaraan: "
        )

        kendaraan = self.cari_kendaraan(
            id_kendaraan
        )

        if kendaraan is None:

            print("Kendaraan tidak ditemukan.")

            return

        if not kendaraan.cek_ketersediaan():

            print("Kendaraan sedang disewa.")

            return

        try:

            lama_sewa = int(
                input("Lama sewa (hari): ")
            )

        except ValueError:

            print("Input harus berupa angka.")

            return

        if lama_sewa <= 0:

            print("Lama sewa harus lebih dari 0.")

            return

        rental = Rental(
            f"R{self.nomor_rental:03d}",
            customer
        )

        berhasil = rental.tambah_kendaraan(
            kendaraan,
            lama_sewa
        )

        if not berhasil:
            return

        print(
            f"\nTotal yang harus dibayar: "
            f"Rp{rental.hitung_total():,.0f}"
        )

        print("\nPilihan metode pembayaran:")
        print("1. Cash")
        print("2. Transfer")
        print("3. E-Wallet")

        pilihan = input(
            "Pilih metode: "
        )

        metode_dict = {
            "1": "Cash",
            "2": "Transfer",
            "3": "E-Wallet"
        }

        metode = metode_dict.get(
            pilihan,
            "Cash"
        )

        rental.proses_pembayaran(
            metode
        )

        rental.tampilkan_struk()

        self.daftar_rental.append(rental)

        self.nomor_rental += 1

        print(
            "\nRental berhasil dibuat."
        )

    # PENGEMBALIAN

    def kembalikan_kendaraan(self):

        print("\n===== PENGEMBALIAN KENDARAAN =====")

        if not self.daftar_rental:

            print("Belum ada transaksi rental.")

            return

        print("\nDaftar Rental:")

        for rental in self.daftar_rental:

            kendaraan = rental.detail_rental.kendaraan

            print(
                f"{rental.id_rental} | "
                f"{rental.customer.nama} | "
                f"{kendaraan.merk} "
                f"{kendaraan.model} | "
                f"{kendaraan.status}"
            )

        id_rental = input(
            "\nMasukkan ID Rental: "
        )

        rental_ditemukan = None

        for rental in self.daftar_rental:

            if rental.id_rental.lower() == id_rental.lower():

                rental_ditemukan = rental

                break

        if rental_ditemukan is None:

            print("Rental tidak ditemukan.")

            return

        kendaraan = (
            rental_ditemukan
            .detail_rental
            .kendaraan
        )

        kendaraan.kembalikan()

        print(
            f"\nKendaraan "
            f"{kendaraan.merk} "
            f"{kendaraan.model} "
            f"berhasil dikembalikan."
        )

    # RIWAYAT RENTAL

    def tampilkan_riwayat(self):

        print("\n===== RIWAYAT RENTAL =====")

        if not self.daftar_rental:

            print("Belum ada transaksi.")

            return

        for rental in self.daftar_rental:

            kendaraan = (
                rental.detail_rental.kendaraan
            )

            print("-" * 55)

            print(
                f"ID Rental : {rental.id_rental}"
            )

            print(
                f"Customer  : {rental.customer.nama}"
            )

            print(
                f"Kendaraan : "
                f"{kendaraan.merk} "
                f"{kendaraan.model}"
            )

            print(
                f"Durasi    : "
                f"{rental.detail_rental.lama_sewa} hari"
            )

            print(
                f"Total     : "
                f"Rp{rental.hitung_total():,.0f}"
            )

            print(
                f"Pembayaran: "
                f"{rental.pembayaran.status}"
            )

    # STATISTIK

    def tampilkan_statistik(self):

        print("\n===== STATISTIK SISTEM =====")

        print(
            f"Total kendaraan : "
            f"{Kendaraan.jumlah_kendaraan}"
        )

        print(
            f"Total customer  : "
            f"{Customer.jumlah_customer}"
        )

        print(
            f"Total rental    : "
            f"{len(self.daftar_rental)}"
        )

# 10. DATA AWAL

def isi_data_awal(sistem):

    # CUSTOMER
    customer1 = Customer(
        "C001",
        "Budi",
        "081234567890",
        "Semarang"
    )

    customer2 = Customer(
        "C002",
        "Siti",
        "082345678901",
        "Demak"
    )

    sistem.daftar_customer.extend(
        [customer1, customer2]
    )

    # MOBIL
    mobil1 = Mobil(
        "K001",
        "Toyota",
        "Avanza",
        2023,
        350000,
        7
    )

    mobil2 = Mobil(
        "K002",
        "Honda",
        "Brio",
        2022,
        300000,
        5
    )

    # MOTOR
    motor1 = Motor(
        "K003",
        "Honda",
        "Vario",
        2023,
        100000,
        "Matic"
    )

    motor2 = Motor(
        "K004",
        "Yamaha",
        "NMAX",
        2022,
        150000,
        "Matic"
    )

    sistem.daftar_kendaraan.extend(
        [
            mobil1,
            mobil2,
            motor1,
            motor2
        ]
    )

# 11. MENU UTAMA

def menu_utama():

    sistem = SistemRental()

    isi_data_awal(sistem)

    while True:

        print("\n")
        print("=" * 60)
        print("          SISTEM RENTAL KENDARAAN")
        print("=" * 60)

        print("1. Tampilkan Kendaraan")
        print("2. Tambah Mobil")
        print("3. Tambah Motor")
        print("4. Tampilkan Customer")
        print("5. Tambah Customer")
        print("6. Buat Rental")
        print("7. Kembalikan Kendaraan")
        print("8. Riwayat Rental")
        print("9. Statistik Sistem")
        print("0. Keluar")

        print("=" * 60)

        pilihan = input(
            "Pilih menu: "
        )

        if pilihan == "1":

            sistem.tampilkan_kendaraan()

        elif pilihan == "2":

            sistem.tambah_mobil()

        elif pilihan == "3":

            sistem.tambah_motor()

        elif pilihan == "4":

            sistem.tampilkan_customer()

        elif pilihan == "5":

            sistem.tambah_customer()

        elif pilihan == "6":

            sistem.buat_rental()

        elif pilihan == "7":

            sistem.kembalikan_kendaraan()

        elif pilihan == "8":

            sistem.tampilkan_riwayat()

        elif pilihan == "9":

            sistem.tampilkan_statistik()

        elif pilihan == "0":

            print(
                "\nTerima kasih telah menggunakan "
                "Sistem Rental Kendaraan."
            )

            break

        else:

            print(
                "\nPilihan tidak tersedia."
            )



menu_utama()



          SISTEM RENTAL KENDARAAN
1. Tampilkan Kendaraan
2. Tambah Mobil
3. Tambah Motor
4. Tampilkan Customer
5. Tambah Customer
6. Buat Rental
7. Kembalikan Kendaraan
8. Riwayat Rental
9. Statistik Sistem
0. Keluar
Pilih menu: 4

===== DAFTAR CUSTOMER =====
----------------------------------------

ID Customer : C001
Nama        : Budi
No. HP      : 081234567890
Alamat      : Semarang

----------------------------------------

ID Customer : C002
Nama        : Siti
No. HP      : 082345678901
Alamat      : Demak



          SISTEM RENTAL KENDARAAN
1. Tampilkan Kendaraan
2. Tambah Mobil
3. Tambah Motor
4. Tampilkan Customer
5. Tambah Customer
6. Buat Rental
7. Kembalikan Kendaraan
8. Riwayat Rental
9. Statistik Sistem
0. Keluar
Pilih menu: 5

===== TAMBAH CUSTOMER =====
Nama       : rehan
No. HP     : 0938279
Alamat     : 

Customer berhasil ditambahkan dengan ID C003.


          SISTEM RENTAL KENDARAAN
1. Tampilkan Kendaraan
2. Tambah Mobil
3. Tambah Motor
4. Tampilkan Customer
5. 

KeyboardInterrupt: Interrupted by user